In [1]:
import os
import json
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np

# ====== AJUSTE AQUI ======
REAL_ROOT  = Path("DATASET54000")
SYNTH_ROOT = Path("DDPM-f64")
OUTPUT_DIR = Path("FAISS-64-54K-RESULTS")

K = 20
BATCH_SIZE = 128
THRESHOLD_QUANTILE_REAL = 0.95

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


OUTPUT_DIR: /tf/2025/FAISS-64-54K-RESULTS


In [2]:
import torch
from PIL import Image
from torchvision import transforms

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# Este é o ponto crítico: usar o MESMO extrator da sua biblioteca de FID 64.
# Aqui usamos torchmetrics (FrechetInceptionDistance) como "feature backend".
try:
    from torchmetrics.image.fid import FrechetInceptionDistance
except Exception as e:
    raise RuntimeError(
        "Não consegui importar torchmetrics.image.fid.FrechetInceptionDistance. "
        "Instale torchmetrics ou diga qual biblioteca você está usando para o FID 64."
    ) from e

# Cria objeto FID e usa a rede interna para extrair features
fid_metric = FrechetInceptionDistance(feature=2048, normalize=True).to(DEVICE)
fid_metric.eval()

# A rede interna (mudou de nome entre versões). Tentamos as opções comuns:
if hasattr(fid_metric, "inception"):
    inception_net = fid_metric.inception
elif hasattr(fid_metric, "feature_network"):
    inception_net = fid_metric.feature_network
else:
    # fallback: tenta achar módulo do inception
    inception_net = None
    for name, mod in fid_metric.named_modules():
        if "inception" in name.lower():
            inception_net = mod
            break
    if inception_net is None:
        raise RuntimeError("Não encontrei a rede Inception dentro do objeto FID do torchmetrics.")

inception_net.eval().to(DEVICE)

# IMPORTANT: Aqui a imagem é carregada em 64x64 (como seu DDPM gera e como você disse que usa no FID 64)
to_uint8_64 = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),  # float 0..1
])

@torch.no_grad()
def extract_features_batch_64(img_paths: List[Path]) -> np.ndarray:
    """
    Converte imagens para uint8 64x64 e extrai features pelo mesmo backbone do FID 64.
    Retorna [B, 2048] float32.
    """
    batch = []
    for p in img_paths:
        img = Image.open(p).convert("RGB")
        t = to_uint8_64(img)                     # [3,64,64] float 0..1
        t = (t * 255.0).clamp(0, 255).byte()     # uint8 0..255
        batch.append(t)

    x = torch.stack(batch, dim=0).to(DEVICE)     # [B,3,64,64] uint8

    # torchmetrics FID backbone geralmente aceita uint8 e faz o preprocess interno quando normalize=True
    feats = inception_net(x)
    if isinstance(feats, (tuple, list)):
        feats = feats[0]
    feats = feats.detach().cpu().numpy().astype(np.float32)
    return feats


DEVICE: cuda


In [3]:
import faiss  # pip install faiss-cpu

def list_classes(root: Path) -> List[str]:
    classes = [d.name for d in root.iterdir() if d.is_dir()]
    classes.sort()
    return classes

def list_images(class_dir: Path) -> List[Path]:
    paths = []
    for p in class_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            paths.append(p)
    paths.sort()
    return paths

def batched(lst: List[Path], bs: int):
    for i in range(0, len(lst), bs):
        yield lst[i:i+bs]

def extract_features_for_dir_64(class_dir: Path, batch_size: int) -> Tuple[np.ndarray, List[str]]:
    paths = list_images(class_dir)
    if not paths:
        raise ValueError(f"Nenhuma imagem em {class_dir}")
    feats_all = []
    for chunk in batched(paths, batch_size):
        feats_all.append(extract_features_batch_64(chunk))
    feats = np.vstack(feats_all).astype(np.float32)
    return feats, [str(p) for p in paths]

def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def save_numpy(path: Path, arr: np.ndarray):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(str(path), arr)


In [4]:
def build_faiss_index(real_feats: np.ndarray):
    d = real_feats.shape[1]
    index = faiss.IndexFlatL2(d)
    index.add(real_feats)
    return index

def knn_mean_distance(index, query_feats: np.ndarray, k: int) -> np.ndarray:
    # IndexFlatL2 retorna L2^2
    D2, _ = index.search(query_feats, k)
    D = np.sqrt(np.maximum(D2, 0.0))
    return D.mean(axis=1)

def knn_mean_distance_excluding_self(index, real_feats: np.ndarray, k: int) -> np.ndarray:
    D2, _ = index.search(real_feats, k + 1)
    D = np.sqrt(np.maximum(D2, 0.0))
    return D[:, 1:].mean(axis=1)

def quality_from_mean_dist(mean_dist: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + mean_dist)

def threshold_from_real(mean_dist_real: np.ndarray, q: float) -> float:
    return float(np.quantile(mean_dist_real, q))


In [5]:
classes_real = list_classes(REAL_ROOT)
classes_synth = list_classes(SYNTH_ROOT)
classes = sorted(set(classes_real) & set(classes_synth))

print("Classes:", classes)

summary = {}

for cls in classes:
    print(f"\n=== Classe: {cls} (64) ===")
    out_dir = OUTPUT_DIR / cls
    out_dir.mkdir(parents=True, exist_ok=True)

    real_dir = REAL_ROOT / cls
    synth_dir = SYNTH_ROOT / cls

    # 1) Features (64) — reais e sintéticas (sintéticas só para stats/ranking)
    real_feats, real_paths = extract_features_for_dir_64(real_dir, batch_size=BATCH_SIZE)
    synth_feats, synth_paths = extract_features_for_dir_64(synth_dir, batch_size=BATCH_SIZE)

    print("real_feats:", real_feats.shape, "synth_feats:", synth_feats.shape)

    # 2) Índice FAISS com reais
    index = build_faiss_index(real_feats)

    # 3) Distâncias kNN
    mean_dist_real  = knn_mean_distance_excluding_self(index, real_feats,  k=K)
    mean_dist_synth = knn_mean_distance(index, synth_feats, k=K)

    # 4) Limiar por quantil dos reais
    thr_dist = threshold_from_real(mean_dist_real, THRESHOLD_QUANTILE_REAL)
    thr_quality = float(1.0 / (1.0 + thr_dist))

    # 5) Salvar artefatos do “modelo” da classe
    faiss.write_index(index, str(out_dir / "faiss_index_l2.index"))
    save_numpy(out_dir / "real_features.npy", real_feats)
    save_json(out_dir / "real_paths.json", {"paths": real_paths})

    meta = {
        "class_name": cls,
        "k": K,
        "feature_backend": "torchmetrics_FID_inception (input 64x64 uint8)",
        "threshold_quantile_real": THRESHOLD_QUANTILE_REAL,
        "threshold_dist_mean_knn": thr_dist,
        "threshold_quality": thr_quality,
        "n_real": int(real_feats.shape[0]),
        "n_synth": int(synth_feats.shape[0]),
        "real_mean_dist_stats": {
            "min": float(np.min(mean_dist_real)),
            "p50": float(np.quantile(mean_dist_real, 0.50)),
            "p90": float(np.quantile(mean_dist_real, 0.90)),
            "p95": float(np.quantile(mean_dist_real, 0.95)),
            "max": float(np.max(mean_dist_real)),
        },
        "synth_mean_dist_stats": {
            "min": float(np.min(mean_dist_synth)),
            "p50": float(np.quantile(mean_dist_synth, 0.50)),
            "p90": float(np.quantile(mean_dist_synth, 0.90)),
            "p95": float(np.quantile(mean_dist_synth, 0.95)),
            "max": float(np.max(mean_dist_synth)),
        },
    }
    save_json(out_dir / "filter_meta.json", meta)

    # opcional: top-50 sintéticas pela métrica (para inspeção)
    quality_synth = quality_from_mean_dist(mean_dist_synth)
    sorted_idx = np.argsort(quality_synth)[::-1]
    top = [{
        "path": synth_paths[int(i)],
        "mean_dist": float(mean_dist_synth[int(i)]),
        "quality": float(quality_synth[int(i)]),
        "passes_threshold": bool(mean_dist_synth[int(i)] <= thr_dist)
    } for i in sorted_idx[:50]]
    save_json(out_dir / "top50_synth_rank.json", {"top50": top})

    summary[cls] = {"threshold_dist": thr_dist, "threshold_quality": thr_quality}

save_json(OUTPUT_DIR / "summary.json", summary)
print("\nConcluído. Filtros 64 salvos em:", OUTPUT_DIR.resolve())


Classes: ['BULKCARRIER', 'CONTAINERSHIP', 'GENERALCARGO', 'OILPRODUCTSTANKER', 'PASSENGERSSHIP', 'TANKER', 'TRAWLER', 'TUG', 'VEHICLESCARRIER', 'YACHT']

=== Classe: BULKCARRIER (64) ===


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/conv.py:459: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,


real_feats: (5400, 2048) synth_feats: (5400, 2048)

=== Classe: CONTAINERSHIP (64) ===
real_feats: (5400, 2048) synth_feats: (5400, 2048)

=== Classe: GENERALCARGO (64) ===
real_feats: (5400, 2048) synth_feats: (5400, 2048)

=== Classe: OILPRODUCTSTANKER (64) ===
real_feats: (5400, 2048) synth_feats: (5400, 2048)

=== Classe: PASSENGERSSHIP (64) ===
real_feats: (5400, 2048) synth_feats: (5400, 2048)

=== Classe: TANKER (64) ===
real_feats: (5400, 2048) synth_feats: (5400, 2048)

=== Classe: TRAWLER (64) ===
real_feats: (5400, 2048) synth_feats: (5400, 2048)

=== Classe: TUG (64) ===
real_feats: (5400, 2048) synth_feats: (5400, 2048)

=== Classe: VEHICLESCARRIER (64) ===
real_feats: (5400, 2048) synth_feats: (5400, 2048)

=== Classe: YACHT (64) ===
real_feats: (5400, 2048) synth_feats: (5400, 2048)

Concluído. Filtros 64 salvos em: /tf/2025/FAISS-64-54K-RESULTS
